# 04 - Information Theory for ML

**Goal:** Implement core information-theoretic quantities from scratch and connect them to ML concepts (loss functions, divergences, feature selection).

**Contents:**
1. Entropy, joint entropy, conditional entropy
2. KL divergence, cross-entropy, JS divergence
3. Mutual information
4. Connections to ML: cross-entropy loss, KL direction asymmetry, decision trees

---

### Key Formulas

| Quantity | Formula | Interpretation |
|----------|---------|---------------|
| Entropy | $H(X) = -\sum p(x) \log p(x)$ | Uncertainty / expected surprise |
| Joint entropy | $H(X,Y) = -\sum_{x,y} p(x,y) \log p(x,y)$ | Total uncertainty |
| Conditional entropy | $H(Y|X) = H(X,Y) - H(X)$ | Remaining uncertainty in Y after observing X |
| KL divergence | $D_{KL}(p\|q) = \sum p(x) \log \frac{p(x)}{q(x)}$ | "Extra bits" from using q instead of p |
| Cross-entropy | $H(p,q) = -\sum p(x) \log q(x) = H(p) + D_{KL}(p\|q)$ | Expected bits using code from q on data from p |
| JS divergence | $D_{JS}(p\|q) = \frac{1}{2}D_{KL}(p\|m) + \frac{1}{2}D_{KL}(q\|m)$ | Symmetric, bounded divergence ($m = \frac{p+q}{2}$) |
| Mutual information | $I(X;Y) = H(X) - H(X|Y) = D_{KL}(p(x,y)\|p(x)p(y))$ | Shared information between X and Y |

All logs are natural (base $e$, units: nats) unless stated otherwise. For bits, use $\log_2$.

**Resources:**
- Cover & Thomas, "Elements of Information Theory" (the bible)
- MacKay, "Information Theory, Inference, and Learning Algorithms" (free online)
- Bishop PRML Ch. 1.6

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Entropy

$H(X) = -\sum_{x} p(x) \log p(x)$

Properties:
- $H(X) \geq 0$
- $H(X) = 0$ iff $X$ is deterministic
- Maximized by the uniform distribution
- For binary: $H(p) = -p\log p - (1-p)\log(1-p)$

In [ ]:
def entropy(p):
    """Shannon entropy of a discrete distribution.
    p: array of probabilities (must sum to 1)
    """
    p = np.asarray(p, dtype=np.float64)
    # Filter out zero probabilities (0 * log(0) = 0 by convention)
    p = p[p > 0]
    return -np.sum(p * np.log(p))

def binary_entropy(p):
    """Entropy of Bernoulli(p)."""
    if p == 0 or p == 1:
        return 0.0
    return -p * np.log(p) - (1 - p) * np.log(1 - p)

# Verify
print(f"Fair coin:   H = {entropy([0.5, 0.5]):.4f} nats = {entropy([0.5, 0.5]) / np.log(2):.4f} bits")
print(f"Biased 0.9:  H = {entropy([0.9, 0.1]):.4f} nats")
print(f"Certain:     H = {entropy([1.0, 0.0]):.4f} nats")
print(f"Fair die:    H = {entropy(np.ones(6)/6):.4f} nats = {entropy(np.ones(6)/6)/np.log(2):.4f} bits")
print(f"  (expected: log(6) = {np.log(6):.4f} nats = {np.log2(6):.4f} bits)")

In [ ]:
# Visualize: Binary entropy as function of p
p_range = np.linspace(0.001, 0.999, 500)
h_range = [binary_entropy(p) for p in p_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_range, h_range, 'b-', linewidth=2)
ax.axhline(np.log(2), color='gray', linestyle=':', alpha=0.5, label=f'ln(2) = {np.log(2):.4f}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('p (probability of heads)')
ax.set_ylabel('H(p) in nats')
ax.set_title('Binary Entropy: Maximum Uncertainty at p=0.5')
ax.legend()
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate('Maximum entropy\n(most uncertain)', xy=(0.5, np.log(2)), xytext=(0.7, 0.5),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')
ax.annotate('Certain outcome\n(no surprise)', xy=(0.05, binary_entropy(0.05)), xytext=(0.15, 0.15),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green')

plt.tight_layout()
plt.show()

## Joint and Conditional Entropy

In [ ]:
def joint_entropy(pxy):
    """Joint entropy H(X,Y) from joint distribution table.
    pxy: 2D array where pxy[i,j] = P(X=i, Y=j)
    """
    pxy = np.asarray(pxy, dtype=np.float64)
    pxy_flat = pxy.flatten()
    pxy_flat = pxy_flat[pxy_flat > 0]
    return -np.sum(pxy_flat * np.log(pxy_flat))

def conditional_entropy(pxy):
    """Conditional entropy H(Y|X) = H(X,Y) - H(X).
    pxy: 2D array where pxy[i,j] = P(X=i, Y=j)
    """
    px = pxy.sum(axis=1)  # marginal of X
    return joint_entropy(pxy) - entropy(px)

# Example: weather (X) and umbrella (Y)
# X: 0=sunny, 1=rainy
# Y: 0=no umbrella, 1=umbrella
pxy = np.array([
    [0.45, 0.05],   # P(sunny, no umbrella), P(sunny, umbrella)
    [0.05, 0.45],   # P(rainy, no umbrella), P(rainy, umbrella)
])

px = pxy.sum(axis=1)  # P(sunny)=0.5, P(rainy)=0.5
py = pxy.sum(axis=0)  # P(no umbrella)=0.5, P(umbrella)=0.5

print(f"H(Weather)         = {entropy(px):.4f} nats")
print(f"H(Umbrella)        = {entropy(py):.4f} nats")
print(f"H(Weather,Umbrella)= {joint_entropy(pxy):.4f} nats")
print(f"H(Umbrella|Weather)= {conditional_entropy(pxy):.4f} nats")
print(f"")
print(f"Chain rule check: H(X,Y) = H(X) + H(Y|X)")
print(f"  {joint_entropy(pxy):.4f} = {entropy(px):.4f} + {conditional_entropy(pxy):.4f} = {entropy(px) + conditional_entropy(pxy):.4f}")
print(f"")
print("Knowing the weather reduces umbrella uncertainty:")
print(f"  H(Y) = {entropy(py):.4f} -> H(Y|X) = {conditional_entropy(pxy):.4f}")
print(f"  Reduction = {entropy(py) - conditional_entropy(pxy):.4f} nats (this is mutual information)")

## 2. KL Divergence, Cross-Entropy, JS Divergence

In [ ]:
def kl_divergence(p, q):
    """KL(p || q) = sum p(x) log(p(x)/q(x)).
    
    Not symmetric! KL(p||q) != KL(q||p).
    Requires: q(x) > 0 wherever p(x) > 0 (absolute continuity).
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    # Only sum over non-zero p entries
    mask = p > 0
    if np.any(q[mask] <= 0):
        return float('inf')
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def cross_entropy(p, q):
    """H(p, q) = -sum p(x) log q(x) = H(p) + KL(p||q)."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    mask = p > 0
    return -np.sum(p[mask] * np.log(q[mask]))

def js_divergence(p, q):
    """Jensen-Shannon divergence: symmetric, bounded [0, ln(2)].
    JS(p||q) = 0.5 * KL(p||m) + 0.5 * KL(q||m), where m = (p+q)/2
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    m = 0.5 * (p + q)
    return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)

# Verify relationships
p = np.array([0.4, 0.3, 0.2, 0.1])
q = np.array([0.25, 0.25, 0.25, 0.25])  # uniform

print(f"H(p)        = {entropy(p):.4f}")
print(f"KL(p||q)    = {kl_divergence(p, q):.4f}")
print(f"H(p,q)      = {cross_entropy(p, q):.4f}")
print(f"H(p)+KL     = {entropy(p) + kl_divergence(p, q):.4f}  (should match H(p,q))")
print(f"")
print(f"Asymmetry:")
print(f"KL(p||q)    = {kl_divergence(p, q):.4f}")
print(f"KL(q||p)    = {kl_divergence(q, p):.4f}")
print(f"")
print(f"JS(p||q)    = {js_divergence(p, q):.4f}")
print(f"JS(q||p)    = {js_divergence(q, p):.4f}  (symmetric!)")

### KL Divergence Asymmetry: Why Direction Matters

This is a critical concept for ML:

- **$D_{KL}(p \| q)$** ("forward KL", "moment matching"): Penalizes $q$ heavily where $p > 0$ but $q \approx 0$. Forces $q$ to cover all modes of $p$ => **mode-covering** behavior. This is what we minimize when training discriminative models (cross-entropy loss).

- **$D_{KL}(q \| p)$** ("reverse KL", "mode seeking"): Penalizes $q$ where $q > 0$ but $p \approx 0$. Allows $q$ to ignore some modes of $p$ => **mode-seeking** behavior. This is what variational inference minimizes (ELBO).

In [ ]:
# Demonstrate KL asymmetry with a bimodal target and unimodal approximation
x = np.linspace(-6, 6, 1000)
dx = x[1] - x[0]

# Bimodal target
p_vals = 0.5 * np.exp(-0.5 * ((x - 2) / 0.7)**2) + 0.5 * np.exp(-0.5 * ((x + 2) / 0.7)**2)
p_vals /= np.sum(p_vals * dx)  # normalize

# Two candidate approximations
# q1: covers both modes (wide Gaussian centered at 0) -- minimizes KL(p||q)
q1_vals = np.exp(-0.5 * (x / 2.0)**2)
q1_vals /= np.sum(q1_vals * dx)

# q2: focuses on one mode -- minimizes KL(q||p)
q2_vals = np.exp(-0.5 * ((x - 2) / 0.7)**2)
q2_vals /= np.sum(q2_vals * dx)

# Compute KLs (discrete approximation)
eps = 1e-10
kl_pq1 = np.sum(p_vals * np.log((p_vals + eps) / (q1_vals + eps)) * dx)
kl_pq2 = np.sum(p_vals * np.log((p_vals + eps) / (q2_vals + eps)) * dx)
kl_q1p = np.sum(q1_vals * np.log((q1_vals + eps) / (p_vals + eps)) * dx)
kl_q2p = np.sum(q2_vals * np.log((q2_vals + eps) / (p_vals + eps)) * dx)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(x, p_vals, alpha=0.3, color='black', label='p (bimodal target)')
ax.plot(x, p_vals, 'k-', linewidth=2)
ax.plot(x, q1_vals, 'b-', linewidth=2, label=f'q1 (mode-covering)')
ax.plot(x, q2_vals, 'r-', linewidth=2, label=f'q2 (mode-seeking)')
ax.set_title(f'Forward KL: KL(p||q) -- penalizes missing modes\n'
             f'KL(p||q1)={kl_pq1:.3f}, KL(p||q2)={kl_pq2:.3f}\n'
             f'q1 wins (covers both modes)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.fill_between(x, p_vals, alpha=0.3, color='black', label='p (bimodal target)')
ax.plot(x, p_vals, 'k-', linewidth=2)
ax.plot(x, q1_vals, 'b-', linewidth=2, label=f'q1 (mode-covering)')
ax.plot(x, q2_vals, 'r-', linewidth=2, label=f'q2 (mode-seeking)')
ax.set_title(f'Reverse KL: KL(q||p) -- penalizes placing mass where p is low\n'
             f'KL(q1||p)={kl_q1p:.3f}, KL(q2||p)={kl_q2p:.3f}\n'
             f'q2 wins (snaps to one mode)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### KL Divergence Between Two Gaussians (Closed Form + Empirical)

For $p = \mathcal{N}(\mu_1, \sigma_1^2)$ and $q = \mathcal{N}(\mu_2, \sigma_2^2)$:

$$D_{KL}(p \| q) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

This formula is heavily used in VAEs (regularization term in ELBO).

In [ ]:
def kl_gaussian(mu1, sigma1, mu2, sigma2):
    """Closed-form KL(N(mu1,sigma1^2) || N(mu2,sigma2^2))."""
    return (np.log(sigma2 / sigma1) + 
            (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2) - 
            0.5)

def kl_gaussian_empirical(mu1, sigma1, mu2, sigma2, n_samples=100000):
    """Empirical KL via Monte Carlo: E_p[log p(x) - log q(x)]."""
    rng = np.random.default_rng(42)
    samples = rng.normal(mu1, sigma1, n_samples)
    log_p = -0.5 * np.log(2 * np.pi * sigma1**2) - (samples - mu1)**2 / (2 * sigma1**2)
    log_q = -0.5 * np.log(2 * np.pi * sigma2**2) - (samples - mu2)**2 / (2 * sigma2**2)
    return np.mean(log_p - log_q)

# Test cases
test_cases = [
    (0, 1, 0, 1, "Same distribution"),
    (0, 1, 1, 1, "Shifted mean"),
    (0, 1, 0, 2, "Wider q"),
    (0, 2, 0, 1, "Narrower q"),
    (1, 0.5, -1, 2, "Different everything"),
]

print(f"{'Description':25s} | {'Closed-form':>12s} | {'Empirical':>12s}")
print("-" * 60)
for mu1, s1, mu2, s2, desc in test_cases:
    kl_cf = kl_gaussian(mu1, s1, mu2, s2)
    kl_emp = kl_gaussian_empirical(mu1, s1, mu2, s2)
    print(f"{desc:25s} | {kl_cf:12.6f} | {kl_emp:12.6f}")

In [ ]:
# Visualize how KL changes as we vary the parameters of q
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fix p = N(0,1), vary mu of q
mu_range = np.linspace(-3, 3, 200)
kl_vs_mu = [kl_gaussian(0, 1, mu, 1) for mu in mu_range]
axes[0].plot(mu_range, kl_vs_mu, 'b-', linewidth=2)
axes[0].set_xlabel(r'$\mu_q$')
axes[0].set_ylabel(r'$D_{KL}(p \| q)$')
axes[0].set_title(r'KL divergence as $\mu_q$ varies (p=N(0,1), q=N($\mu_q$,1))')
axes[0].grid(True, alpha=0.3)

# Fix p = N(0,1), vary sigma of q
sigma_range = np.linspace(0.2, 5, 200)
kl_vs_sigma = [kl_gaussian(0, 1, 0, s) for s in sigma_range]
axes[1].plot(sigma_range, kl_vs_sigma, 'r-', linewidth=2)
axes[1].axvline(1.0, color='gray', linestyle=':', label=r'$\sigma_q = \sigma_p = 1$')
axes[1].set_xlabel(r'$\sigma_q$')
axes[1].set_ylabel(r'$D_{KL}(p \| q)$')
axes[1].set_title(r'KL divergence as $\sigma_q$ varies (p=N(0,1), q=N(0,$\sigma_q^2$))')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Mutual Information

$I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X,Y)$

Equivalently: $I(X;Y) = D_{KL}(p(x,y) \| p(x)p(y))$

MI measures how much knowing $Y$ tells you about $X$ (and vice versa). It's zero iff $X$ and $Y$ are independent.

In [ ]:
def mutual_information(pxy):
    """Mutual information I(X;Y) from joint distribution.
    pxy: 2D array where pxy[i,j] = P(X=i, Y=j)
    """
    px = pxy.sum(axis=1)  # marginal X
    py = pxy.sum(axis=0)  # marginal Y
    return entropy(px) + entropy(py) - joint_entropy(pxy)

def mutual_information_kl(pxy):
    """MI as KL(p(x,y) || p(x)p(y)). Should give same result."""
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    # Outer product = p(x)p(y) assuming independence
    pxpy = np.outer(px, py)
    return kl_divergence(pxy.flatten(), pxpy.flatten())

# Test with the weather/umbrella example
print(f"MI (entropy method): {mutual_information(pxy):.4f}")
print(f"MI (KL method):      {mutual_information_kl(pxy):.4f}")
print()

# Independent case: MI should be 0
pxy_indep = np.outer([0.3, 0.7], [0.4, 0.6])  # independent by construction
print(f"Independent case MI: {mutual_information(pxy_indep):.6f} (should be ~0)")

# Perfect correlation: MI = H(X) = H(Y)
pxy_perfect = np.array([[0.5, 0.0], [0.0, 0.5]])  # X = Y
print(f"Perfect correlation MI: {mutual_information(pxy_perfect):.4f}")
print(f"H(X) = H(Y) = {entropy([0.5, 0.5]):.4f}")

### Estimating MI from Continuous Data

For continuous variables, we discretize (bin) the data to estimate MI. This is a common practical approach, though it's biased for small samples.

In [ ]:
def estimate_mi_binned(x, y, n_bins=30):
    """Estimate mutual information from continuous data using binning."""
    # Create 2D histogram
    hist_2d, x_edges, y_edges = np.histogram2d(x, y, bins=n_bins)
    # Normalize to get joint probability
    pxy = hist_2d / hist_2d.sum()
    
    # Marginals
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    
    # MI
    mi = 0.0
    for i in range(n_bins):
        for j in range(n_bins):
            if pxy[i, j] > 0 and px[i] > 0 and py[j] > 0:
                mi += pxy[i, j] * np.log(pxy[i, j] / (px[i] * py[j]))
    return mi

# Generate correlated Gaussian data with varying correlation
rng = np.random.default_rng(42)
n = 10000

rhos = np.linspace(0, 0.99, 50)
mi_empirical = []
mi_theoretical = []

for rho in rhos:
    # Bivariate normal with correlation rho
    cov = np.array([[1, rho], [rho, 1]])
    data = rng.multivariate_normal([0, 0], cov, n)
    
    mi_emp = estimate_mi_binned(data[:, 0], data[:, 1], n_bins=30)
    mi_empirical.append(mi_emp)
    
    # Theoretical MI for bivariate Gaussian: I = -0.5 * log(1 - rho^2)
    mi_theoretical.append(-0.5 * np.log(1 - rho**2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rhos, mi_theoretical, 'r-', linewidth=2, label='Theoretical: $-\\frac{1}{2}\\ln(1-\\rho^2)$')
ax.plot(rhos, mi_empirical, 'b.', markersize=5, label='Empirical (binned)')
ax.set_xlabel(r'Correlation $\rho$')
ax.set_ylabel('Mutual Information (nats)')
ax.set_title('MI vs Correlation for Bivariate Gaussian')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. MI for Feature Selection

Select features that have the highest mutual information with the target variable. This is a filter-based feature selection method.

In [ ]:
# Generate a classification dataset
rng = np.random.default_rng(42)
n = 2000

# Features: some informative, some noise
# y depends on x0, x1, x2 (informative). x3-x9 are pure noise.
X = rng.standard_normal((n, 10))

# Target: y = sign(x0 + 2*x1 - x2 + noise)
logits = X[:, 0] + 2 * X[:, 1] - X[:, 2] + 0.5 * rng.standard_normal(n)
y = (logits > 0).astype(int)

feature_names = [f'x{i}' for i in range(10)]

def mi_feature_target(x_col, y_col, n_bins=20):
    """Estimate MI between a continuous feature and binary target."""
    # For continuous-discrete MI, bin the continuous variable
    x_binned = np.digitize(x_col, np.linspace(x_col.min(), x_col.max(), n_bins + 1)[1:-1])
    
    # Build joint distribution
    n_x_bins = n_bins
    n_y_vals = len(np.unique(y_col))
    joint = np.zeros((n_x_bins, n_y_vals))
    for xi, yi in zip(x_binned, y_col):
        joint[xi, yi] += 1
    joint /= joint.sum()
    
    return mutual_information(joint)

# Compute MI for each feature
mi_scores = []
for i in range(X.shape[1]):
    mi = mi_feature_target(X[:, i], y)
    mi_scores.append(mi)
    print(f"{feature_names[i]:5s} | MI = {mi:.4f} {'<-- informative' if i < 3 else '(noise)'}")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3' if i < 3 else '#BDBDBD' for i in range(10)]
bars = ax.bar(feature_names, mi_scores, color=colors)
ax.set_ylabel('Mutual Information with Target')
ax.set_title('Feature Selection via Mutual Information\n(Blue = truly informative, Gray = noise)')
ax.grid(True, alpha=0.3, axis='y')

# Add a threshold line
threshold = np.mean(mi_scores) + np.std(mi_scores)
ax.axhline(threshold, color='red', linestyle='--', label=f'Selection threshold ({threshold:.4f})', alpha=0.7)
ax.legend()

plt.tight_layout()
plt.show()

selected = [feature_names[i] for i in range(10) if mi_scores[i] > threshold]
print(f"\nSelected features: {selected}")

## 5. Connection to ML: Cross-Entropy Loss IS KL Divergence

For classification, the cross-entropy loss is:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \sum_c y_{ic} \log \hat{y}_{ic}$$

This is exactly $H(p_{\text{data}}, q_{\text{model}}) = H(p_{\text{data}}) + D_{KL}(p_{\text{data}} \| q_{\text{model}})$.

Since $H(p_{\text{data}})$ is constant, **minimizing cross-entropy loss = minimizing KL divergence between the data distribution and the model**.

In [ ]:
# Demonstrate the equivalence
def softmax(logits):
    e = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Simulate: 3-class classification
np.random.seed(42)
n = 1000
n_classes = 3

# True labels (one-hot)
true_labels = np.random.randint(0, n_classes, n)
y_onehot = np.eye(n_classes)[true_labels]

# Model predictions (logits -> softmax)
logits = np.random.randn(n, n_classes)
probs = softmax(logits)

# Method 1: Standard cross-entropy loss
ce_loss = -np.mean(np.sum(y_onehot * np.log(probs + 1e-10), axis=1))

# Method 2: Compute as H(p) + KL(p||q) over the empirical distributions
# Empirical class frequencies
p_data = np.bincount(true_labels, minlength=n_classes) / n
# Average model prediction per class
q_model = probs.mean(axis=0)

# Per-sample decomposition
H_p = entropy(p_data)
# The per-sample cross-entropy averages to:
per_sample_ce = -np.mean(np.log(probs[np.arange(n), true_labels] + 1e-10))

print(f"Cross-entropy loss:          {ce_loss:.4f}")
print(f"Per-sample CE (equivalent):  {per_sample_ce:.4f}")
print(f"")
print(f"H(p_data) = {H_p:.4f} nats  (entropy of true label distribution)")
print(f"")
print("Key insight: The cross-entropy loss we minimize in training")
print("is exactly the KL divergence (up to the constant H(p)).")
print("A perfect model would achieve loss = H(p) (irreducible entropy).")

In [ ]:
# Show how CE loss decreases as model predictions improve
# Simulate training: gradually improve predictions
temps = np.linspace(5.0, 0.1, 100)  # temperature: high=uniform, low=confident
ce_losses = []
kl_losses = []

for temp in temps:
    # Create "improving" predictions by using true labels with decreasing temperature
    logits_sim = np.zeros((n, n_classes))
    logits_sim[np.arange(n), true_labels] = 1.0 / temp
    probs_sim = softmax(logits_sim)
    
    ce = -np.mean(np.log(probs_sim[np.arange(n), true_labels] + 1e-10))
    ce_losses.append(ce)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(temps)), ce_losses, 'b-', linewidth=2, label='Cross-entropy loss')
ax.axhline(H_p, color='red', linestyle='--', label=f'H(p_data) = {H_p:.4f} (lower bound)', alpha=0.7)
ax.set_xlabel('"Training progress" (model confidence increasing)')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Cross-Entropy Loss Approaches H(p) as Model Improves\n'
             '(The gap above H(p) is exactly KL(p||q))')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Information Gain in Decision Trees = Mutual Information

When a decision tree splits on feature $X$ to predict $Y$:

$$\text{Information Gain} = H(Y) - H(Y|X) = I(X;Y)$$

The best split maximizes information gain (i.e., mutual information between the feature and the target).

In [ ]:
def information_gain(y, x_split):
    """Information gain from splitting y based on binary feature x_split.
    
    IG = H(Y) - H(Y|X) = H(Y) - [P(X=0)*H(Y|X=0) + P(X=1)*H(Y|X=1)]
    """
    n = len(y)
    
    # H(Y) - entropy before split
    _, counts = np.unique(y, return_counts=True)
    h_y = entropy(counts / n)
    
    # H(Y|X) - weighted entropy after split
    h_y_given_x = 0.0
    for val in np.unique(x_split):
        mask = x_split == val
        p_x = mask.sum() / n
        _, counts_subset = np.unique(y[mask], return_counts=True)
        h_y_given_x += p_x * entropy(counts_subset / counts_subset.sum())
    
    return h_y - h_y_given_x

# Demo: simple dataset
# Tennis dataset (play tennis or not based on weather features)
# Outlook: 0=Sunny, 1=Overcast, 2=Rain
outlook  = np.array([0, 0, 1, 2, 2, 2, 1, 0, 0, 2, 0, 1, 1, 2])
humidity = np.array([1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1])  # 0=normal, 1=high
wind     = np.array([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1])  # 0=weak, 1=strong
play     = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0])  # 0=no, 1=yes

ig_outlook  = information_gain(play, outlook)
ig_humidity = information_gain(play, humidity)
ig_wind     = information_gain(play, wind)

print(f"Information Gain (Outlook):   {ig_outlook:.4f} nats")
print(f"Information Gain (Humidity):  {ig_humidity:.4f} nats")
print(f"Information Gain (Wind):      {ig_wind:.4f} nats")
print(f"\nBest first split: {'Outlook' if ig_outlook >= max(ig_humidity, ig_wind) else 'Humidity' if ig_humidity >= ig_wind else 'Wind'}")

fig, ax = plt.subplots(figsize=(8, 4))
features = ['Outlook', 'Humidity', 'Wind']
igs = [ig_outlook, ig_humidity, ig_wind]
colors = ['#4CAF50' if ig == max(igs) else '#90CAF9' for ig in igs]
ax.bar(features, igs, color=colors)
ax.set_ylabel('Information Gain (nats)')
ax.set_title('Decision Tree: Which Feature Gives Most Information?\n(IG = MI between feature and target)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Venn Diagram of Information Quantities

Visualize the relationships between H(X), H(Y), H(X,Y), H(X|Y), H(Y|X), and I(X;Y).

In [ ]:
# Compute all quantities for the weather/umbrella example
px = pxy.sum(axis=1)
py = pxy.sum(axis=0)

H_X = entropy(px)
H_Y = entropy(py)
H_XY = joint_entropy(pxy)
H_X_given_Y = conditional_entropy(pxy.T)  # H(X|Y)
H_Y_given_X = conditional_entropy(pxy)     # H(Y|X)
I_XY = mutual_information(pxy)

print("Information Quantities (Weather/Umbrella):")
print(f"  H(X)    = {H_X:.4f}")
print(f"  H(Y)    = {H_Y:.4f}")
print(f"  H(X,Y)  = {H_XY:.4f}")
print(f"  H(X|Y)  = {H_X_given_Y:.4f}")
print(f"  H(Y|X)  = {H_Y_given_X:.4f}")
print(f"  I(X;Y)  = {I_XY:.4f}")
print()
print("Verify identities:")
print(f"  H(X,Y) = H(X) + H(Y|X) = {H_X:.4f} + {H_Y_given_X:.4f} = {H_X + H_Y_given_X:.4f}")
print(f"  H(X,Y) = H(Y) + H(X|Y) = {H_Y:.4f} + {H_X_given_Y:.4f} = {H_Y + H_X_given_Y:.4f}")
print(f"  I(X;Y) = H(X) - H(X|Y) = {H_X:.4f} - {H_X_given_Y:.4f} = {H_X - H_X_given_Y:.4f}")
print(f"  I(X;Y) = H(X) + H(Y) - H(X,Y) = {H_X:.4f} + {H_Y:.4f} - {H_XY:.4f} = {H_X + H_Y - H_XY:.4f}")

In [ ]:
# Draw Venn diagram using matplotlib circles
fig, ax = plt.subplots(figsize=(10, 6))

# Draw two overlapping circles
from matplotlib.patches import Circle

c1 = Circle((0.35, 0.5), 0.25, fill=False, edgecolor='blue', linewidth=2, label='H(X)')
c2 = Circle((0.65, 0.5), 0.25, fill=False, edgecolor='red', linewidth=2, label='H(Y)')
ax.add_patch(c1)
ax.add_patch(c2)

# Labels
ax.text(0.22, 0.5, f'H(X|Y)\n{H_X_given_Y:.3f}', ha='center', va='center', fontsize=11, color='blue')
ax.text(0.50, 0.5, f'I(X;Y)\n{I_XY:.3f}', ha='center', va='center', fontsize=11, color='purple', fontweight='bold')
ax.text(0.78, 0.5, f'H(Y|X)\n{H_Y_given_X:.3f}', ha='center', va='center', fontsize=11, color='red')

ax.text(0.35, 0.82, f'H(X) = {H_X:.3f}', ha='center', fontsize=12, color='blue', fontweight='bold')
ax.text(0.65, 0.82, f'H(Y) = {H_Y:.3f}', ha='center', fontsize=12, color='red', fontweight='bold')
ax.text(0.50, 0.15, f'H(X,Y) = {H_XY:.3f}', ha='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, 1)
ax.set_ylim(0.1, 0.9)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Information Theory Venn Diagram', fontsize=14)
plt.tight_layout()
plt.show()

---

## Interview Notes and Quick Reference

### Cross-Entropy Loss = KL Divergence (+ Constant)

$$\text{CE Loss} = H(p_{\text{data}}, q_\theta) = H(p_{\text{data}}) + D_{KL}(p_{\text{data}} \| q_\theta)$$

Since $H(p_{\text{data}})$ is constant w.r.t. model params, minimizing CE loss = minimizing KL divergence.

### Why KL(p||q) Not KL(q||p)?

In supervised learning, we minimize $D_{KL}(p_{\text{data}} \| q_{\text{model}})$ because:
- We sample from $p_{\text{data}}$ (training data)
- The expectation $\mathbb{E}_p[\log q]$ can be estimated from samples
- This forces the model to cover all modes of the data (mode-covering)

In variational inference, we minimize $D_{KL}(q_{\text{approx}} \| p_{\text{posterior}})$ because:
- We can sample from $q$ (the variational approximation)
- We cannot sample from $p$ (the true posterior)
- This leads to mode-seeking behavior (VI may miss modes)

### Decision Trees and Information Gain

- **ID3/C4.5** use information gain (= MI) to select split features
- **CART** uses Gini impurity instead (approximation to entropy: $G = 1 - \sum p_i^2$)
- Gini and entropy-based splits almost always select the same features

### JS Divergence in GANs

The original GAN objective (Goodfellow et al., 2014) implicitly minimizes:

$$2 \cdot D_{JS}(p_{\text{data}} \| p_{\text{generator}}) - \log 4$$

JS divergence is symmetric and bounded, but can saturate when distributions don't overlap (leading to training instability). This motivated Wasserstein GAN (WGAN) which uses Earth Mover's distance instead.

### KL Divergence in VAEs

The ELBO (Evidence Lower Bound):

$$\log p(x) \geq \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) \| p(z))$$

The KL term regularizes the encoder to stay close to the prior $p(z)$. For Gaussian encoder and standard normal prior, this has the closed-form we implemented above.

**Further reading:**
- Cover & Thomas, Ch. 2 (entropy), Ch. 8 (differential entropy)
- Goodfellow et al., "Generative Adversarial Networks" (2014) -- JS connection
- Kingma & Welling, "Auto-Encoding Variational Bayes" (2014) -- KL in VAE